# Assignment 3 — Task 1: Data Preparation & Multimodal LLM Description
**Modality:** Histology / Microscopy — Cell Nuclei
**Student ID:** 24142732

Run this in **Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU).
Run cells **top to bottom, in order** — do not skip any cell.


## 0. Setup — Install Ollama in Colab

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(6)
print("Ollama server started.")


In [ ]:
# Pull the multimodal model (~7-8 GB, can take several minutes)
!ollama pull llama3.2-vision


In [ ]:
!pip install -q ollama scikit-image opencv-python-headless matplotlib pillow pandas


In [ ]:
import os


## 1. Download and unzip the dataset

In [ ]:
import zipfile, urllib.request

DATA_URL = "https://github.com/Nickolay-K/Assingnment-3-dataset/raw/main/nuclei_dataset.zip"
ZIP_PATH = "/content/nuclei_dataset.zip"
EXTRACT_DIR = "/content"   # zip already contains a top-level "nuclei_dataset" folder

if not os.path.exists(ZIP_PATH):
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Downloaded.")

if not os.path.exists("/content/nuclei_dataset"):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")

BASE_DIR = "/content/nuclei_dataset"
print("Contents of BASE_DIR:", os.listdir(BASE_DIR))


**Confirmed dataset structure** (already checked — no need to guess):
```
nuclei_dataset/
  train/   images/  masks/  labels/   (80 images)
  val/     images/  masks/  labels/   (20 images)
  test/    images/  masks/  labels/   (12 images)
  test_corrupted/  images/            (4 pre-corrupted images — for the extension task)
  metadata.csv
  README.md
  dataset_summary.json
```

In [ ]:
TRAIN_IMAGE_DIR = os.path.join(BASE_DIR, "train", "images")
TRAIN_MASK_DIR  = os.path.join(BASE_DIR, "train", "masks")

VAL_IMAGE_DIR   = os.path.join(BASE_DIR, "val", "images")
VAL_MASK_DIR    = os.path.join(BASE_DIR, "val", "masks")

TEST_IMAGE_DIR  = os.path.join(BASE_DIR, "test", "images")
TEST_MASK_DIR   = os.path.join(BASE_DIR, "test", "masks")

TEST_CORRUPTED_DIR = os.path.join(BASE_DIR, "test_corrupted", "images")

import glob

def list_images(folder):
    paths = sorted(glob.glob(os.path.join(folder, "*.*")))
    return [p for p in paths if p.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))]

train_image_paths = list_images(TRAIN_IMAGE_DIR)
val_image_paths   = list_images(VAL_IMAGE_DIR)
test_image_paths  = list_images(TEST_IMAGE_DIR)

print(f"Train images: {len(train_image_paths)}")
print(f"Val images:   {len(val_image_paths)}")
print(f"Test images:  {len(test_image_paths)}")
assert len(train_image_paths) > 0, "No images found — check BASE_DIR and folder names above."
print(train_image_paths[:5])

# Use training images for the Task 1 EDA / VLM demo
image_paths = train_image_paths


## 2. Grayscale conversion + resize to 256×256

In [ ]:
import cv2
import numpy as np

OUT_DIR = "/content/processed_256"
os.makedirs(OUT_DIR, exist_ok=True)

def preprocess_image(path, size=256):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img

processed = []
for p in image_paths:
    img = preprocess_image(p)
    out_path = os.path.join(OUT_DIR, os.path.basename(p))
    cv2.imwrite(out_path, img)
    processed.append(out_path)

print(f"Processed {len(processed)} images -> saved in '{OUT_DIR}/'")
assert len(processed) > 0, "No images were processed."


## 3. EDA — sample images + intensity histogram

In [ ]:
import matplotlib.pyplot as plt

n_samples = min(6, len(processed))
sample_paths = processed[:n_samples]

fig, axes = plt.subplots(1, n_samples, figsize=(3*n_samples, 3))
if n_samples == 1:
    axes = [axes]
for ax, p in zip(axes, sample_paths):
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap="gray")
    ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis("off")
plt.suptitle("Sample grayscale nuclei images (256x256)")
plt.tight_layout()
plt.savefig("/content/eda_samples.png", dpi=150)
plt.show()


In [ ]:
# Intensity histogram across the dataset
all_pixels = []
for p in processed:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    all_pixels.append(img.ravel())
all_pixels = np.concatenate(all_pixels)

plt.figure(figsize=(6, 4))
plt.hist(all_pixels, bins=64, color="steelblue")
plt.title("Pixel Intensity Histogram — All Training Images")
plt.xlabel("Pixel intensity (0-255)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("/content/eda_histogram.png", dpi=150)
plt.show()

print(f"Mean intensity: {all_pixels.mean():.2f}, Std: {all_pixels.std():.2f}")


## 4. Multimodal LLM description (llama3.2-vision via Ollama)

Two prompt styles are compared:
- **Naive prompt** — open-ended, no constraints
- **Structured / anchored prompt** — descriptive-only, strict JSON, allows `"uncertain"`


In [ ]:
import ollama
import base64, json

def image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

representative_image = processed[0]
img_b64 = image_to_base64(representative_image)
print("Using:", representative_image)


In [ ]:
NAIVE_PROMPT = "Describe this medical image."

STRUCTURED_PROMPT = """You are an image-description assistant helping a researcher document a microscopy image.
You are NOT a diagnostic tool and must never provide a diagnosis, disease name, or clinical judgement.
Your only job is to describe what is visually present, staying strictly descriptive.

If you are not confident about a field, you MUST write "uncertain" instead of guessing.

Respond with ONLY a valid JSON object, no extra text, no markdown fences, using exactly this schema:
{
  "modality": "<imaging modality, or 'uncertain'>",
  "tissue_type": "<tissue/cell type visible, or 'uncertain'>",
  "notable_features": ["<short phrase>", "<short phrase>", "..."],
  "image_quality": "<one of: good, moderate, poor, uncertain>"
}
"""

def query_vlm(prompt, image_b64, model="llama3.2-vision"):
    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [image_b64],
        }]
    )
    return response["message"]["content"]

naive_output = query_vlm(NAIVE_PROMPT, img_b64)
structured_output = query_vlm(STRUCTURED_PROMPT, img_b64)

print("=== NAIVE PROMPT OUTPUT ===")
print(naive_output)
print("\n=== STRUCTURED PROMPT OUTPUT ===")
print(structured_output)


In [ ]:
# Try to parse the structured output as JSON
def try_parse_json(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json\n", "", 1)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)

parsed, err = try_parse_json(structured_output)
if parsed:
    print("Parsed successfully:")
    print(json.dumps(parsed, indent=2))
else:
    print("JSON parse failed:", err)
    print("Raw output was:", structured_output)


### 4b. Repeated-run variability check

In [ ]:
repeated_outputs = []
for i in range(3):
    out = query_vlm(STRUCTURED_PROMPT, img_b64)
    repeated_outputs.append(out)
    print(f"--- Run {i+1} ---")
    print(out)
    print()


In [ ]:
with open("/content/task1_outputs.json", "w") as f:
    json.dump({
        "representative_image": representative_image,
        "naive_prompt": NAIVE_PROMPT,
        "structured_prompt": STRUCTURED_PROMPT,
        "naive_output": naive_output,
        "structured_output": structured_output,
        "repeated_runs": repeated_outputs,
    }, f, indent=2)

print("Saved /content/task1_outputs.json")
from google.colab import files
files.download("/content/task1_outputs.json")
files.download("/content/eda_samples.png")
files.download("/content/eda_histogram.png")


## Notes for your report (Task 1)
- Paste the **structured prompt** into the report verbatim.
- Use `task1_outputs.json` for naive vs structured comparison + repeated-run evidence.
- Use `eda_samples.png` and `eda_histogram.png` as EDA figures.
